# 1C Predict Future Sales weekly panel for ICDN

Step-by-step construction of the observation unit

`(store, product, week) → (price, units, promo, category)`

We do **not** call `builder.run()`. Each cell invokes one method so we can inspect the intermediate objects.

**Files used:** `sales_train.csv`, `items.csv`, `item_categories.csv`, `shops.csv`.  
**Not used:** `test.csv` (Nov 2015 pairs, no sales/prices), `sample_submission.csv`.

Demand is **gross positive** units, not net of returns. Missing shop-item-weeks are **not** confirmed zeros. `date_block_num` is monthly (`0…33`) and is **not** `week_id`.

`on_promo` is a backward-looking markdown proxy, not observed promotion.

In [1]:
from pathlib import Path
import importlib.util
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

spec = importlib.util.spec_from_file_location(
    "one_c", PROJECT_ROOT / "src" / "1c.py"
)
one_c = importlib.util.module_from_spec(spec)
sys.modules["one_c"] = one_c          
spec.loader.exec_module(one_c)

OneCConfig = one_c.OneCConfig
OneCWeeklyPanelBuilder = one_c.OneCWeeklyPanelBuilder

DATA_DIR = PROJECT_ROOT / "data" / "predict-future-sales-1c"
OUT_DIR = DATA_DIR / "panel"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR    :", DATA_DIR)
print("Exists      :", DATA_DIR.exists())
print("Files       :", sorted(p.name for p in DATA_DIR.glob("*.csv")) if DATA_DIR.exists() else "—")

PROJECT_ROOT: /home/thebigmonster/Github/nn-elasticity-additional-work
DATA_DIR    : /home/thebigmonster/Github/nn-elasticity-additional-work/data/predict-future-sales-1c
Exists      : True
Files       : ['item_categories.csv', 'items.csv', 'sales_train.csv', 'sample_submission.csv', 'shops.csv', 'test.csv']


## 0. Config and builder

Selection rules are stored now but applied only on the first half of complete weeks.  
`target_category_id` stays `None` until we inspect `category_stats`.

In [2]:
config = OneCConfig(
    data_dir=DATA_DIR,
    out_dir=OUT_DIR,
    selection_frac=0.50,
    min_store_week_coverage=0.85,
    n_core_stores=20,
    min_stores=10,
    min_week_coverage=0.70,
    min_unique_prices=8,
    min_price_cv=0.03,
    min_promo_ref_coverage=0.80,
    max_return_rate=0.05,
    n_candidate_skus=30,
    n_skus=10,
    target_category_id=None,
)

builder = OneCWeeklyPanelBuilder(config)

## 1. Load the four tables

Do not load `test.csv`. Shop names are for audit only; we do not parse brand/size from `item_name`.

In [3]:
builder.load_tables()
print(builder.sales.shape, builder.items.shape, builder.categories.shape, builder.shops.shape)
builder.sales.head()

sales (2935849, 6) items (22170, 3)
(2935849, 6) (22170, 3) (84, 2) (60, 2)


,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
0,02.01.2013,0,59,22154,999.0000,1.0000
1,03.01.2013,0,25,2552,899.0000,1.0000
2,05.01.2013,0,25,2552,899.0000,-1.0000
3,06.01.2013,0,25,2554,"1,709.0500",1.0000
4,15.01.2013,0,25,2555,"1,099.0000",1.0000


## 2. Parse dates

Official format is day.month.year (`%d.%m.%Y`).

In [4]:
builder.parse_dates()
print(builder.sales["date"].min(), "→", builder.sales["date"].max())
builder.sales[["date", "date_block_num", "shop_id", "item_id", "item_price", "item_cnt_day"]].head()

2013-01-01 00:00:00 → 2015-10-31 00:00:00


,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
0,2013-01-01,0,18,5823,"2,500.0000",1.0000
1,2013-01-01,0,27,5573,849.0000,1.0000
2,2013-01-01,0,7,1006,399.0000,1.0000
3,2013-01-01,0,19,17707,899.0000,1.0000
4,2013-01-01,0,14,19548,149.0000,1.0000


## 3. Audit before dropping anything

Keep these counts for the appendix. Kaggle: 34 months; `item_cnt_day` is daily units (can be negative).

In [5]:
audit = builder.audit_sales()
pd.Series(audit)

rows                    2935849
shops                        60
items                     21807
months                       34
min_date             2013-01-01
max_date             2015-10-31
negative_units             7356
zero_units                    0
nonpositive_price             1
missing_price                 0
dtype: object


rows                    2935849
shops                        60
items                     21807
months                       34
min_date             2013-01-01
max_date             2015-10-31
negative_units             7356
zero_units                    0
nonpositive_price             1
missing_price                 0
dtype: object

## 4. Split gross sales and returns

$$Q^{+}_{ist}=\sum_{d\in t}\max(q_{isd},0)$$

Returns are written to `1c_returns_audit.parquet` and never enter `log q`.  
No global Kaggle winsorizing (`item_price < 100000`, `item_cnt_day < 1000`).

In [6]:
positive_sales, returns = builder.split_sales_and_returns()
print("positive:", positive_sales.shape, "returns:", returns.shape)
positive_sales.head()

positive rows 2,928,492 | return rows 7,356 (wrote /home/thebigmonster/Github/nn-elasticity-additional-work/data/predict-future-sales-1c/panel/1c_returns_audit.parquet)
positive: (2928492, 7) returns: (7356, 6)


,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,revenue
0,2013-01-01,0,18,5823,"2,500.0000",1.0000,"2,500.0000"
1,2013-01-01,0,27,5573,849.0000,1.0000,849.0000
2,2013-01-01,0,7,1006,399.0000,1.0000,399.0000
3,2013-01-01,0,19,17707,899.0000,1.0000,899.0000
4,2013-01-01,0,14,19548,149.0000,1.0000,149.0000


## 5. Sequential `week_id` (not `date_block_num`)

Monday–Sunday weeks. Truncated first/last weeks are dropped. Then:

`week_start → week_id ∈ {1, 2, …, T}`

In [7]:
positive_sales = builder.build_week_index(positive_sales)

print(builder.week_map.head(10))
print("n complete weeks:", len(builder.week_map))
print("max week_id gap:", np.diff(np.sort(builder.week_map["week_id"])).max())
positive_sales[["date", "week_start", "week_id", "date_block_num"]].head(15)

n_days
6      2
7    146
Name: count, dtype: int64
week_start  week_id
2013-01-07        1
2013-01-14        2
2013-01-21        3
2013-01-28        4
2013-02-04        5
2013-02-11        6
2013-02-18        7
2013-02-25        8 
… 146 complete weeks with positive sales
  week_start  week_id
0 2013-01-07        1
1 2013-01-14        2
2 2013-01-21        3
3 2013-01-28        4
4 2013-02-04        5
5 2013-02-11        6
6 2013-02-18        7
7 2013-02-25        8
8 2013-03-04        9
9 2013-03-11       10
n complete weeks: 146
max week_id gap: 1


,date,week_start,week_id,date_block_num
0,2013-01-07,2013-01-07,1,0
1,2013-01-07,2013-01-07,1,0
2,2013-01-07,2013-01-07,1,0
3,2013-01-07,2013-01-07,1,0
4,2013-01-07,2013-01-07,1,0
5,2013-01-07,2013-01-07,1,0
6,2013-01-07,2013-01-07,1,0
7,2013-01-07,2013-01-07,1,0
8,2013-01-07,2013-01-07,1,0
9,2013-01-07,2013-01-07,1,0


## 6. Daily → weekly units and revenue

$$P_{ist}=\frac{\sum p_{isd}q_{isd}}{\sum q_{isd}},\quad q_{isd}>0$$

Not the mean of `item_price`.

In [8]:
weekly = builder.aggregate_daily_to_weekly(positive_sales)
del positive_sales
print(weekly.shape)
weekly.head()

               price          units
count 2,281,868.0000 2,281,868.0000
mean        836.4838         1.5802
std       1,619.6903         4.3989
min           0.0900         1.0000
1%           49.0000         1.0000
5%          108.4900         1.0000
50%         399.0000         1.0000
95%       2,599.0000         3.0000
99%       5,690.0000         9.0000
max     307,980.0000     1,312.0000
(2281868, 11)


,shop_id,item_id,week_id,week_start,units,revenue,n_sales_days,n_price_points,min_daily_price,max_daily_price,price
0,0,30,6,2013-02-11,15.0000,"3,975.0000",3,1,265.0000,265.0000,265.0000
1,0,30,7,2013-02-18,13.0000,"3,445.0000",5,1,265.0000,265.0000,265.0000
2,0,30,8,2013-02-25,3.0000,795.0000,1,1,265.0000,265.0000,265.0000
3,0,31,6,2013-02-11,7.0000,"3,038.0000",3,1,434.0000,434.0000,434.0000
4,0,31,7,2013-02-18,2.0000,868.0000,2,1,434.0000,434.0000,434.0000


## 7. Weekly returns (audit only)

`return_units` stays on the master so SKUs with a weird return pattern can be screened out. It is not an ICDN column.

In [9]:
weekly = builder.attach_weekly_returns(weekly, returns)
del returns
print("share of rows with returns:", (weekly["return_units"] > 0).mean())
weekly[["shop_id", "item_id", "week_id", "units", "return_units"]].head()

share of rows with returns: 0.0018388443152715231


,shop_id,item_id,week_id,units,return_units
0,0,30,6,15.0000,0.0000
1,0,30,7,13.0000,0.0000
2,0,30,8,3.0000,0.0000
3,0,31,6,7.0000,0.0000
4,0,31,7,2.0000,0.0000


## 8. Product, category, shop names

ICDN `category` = `item_category_name`. `item_name` / `shop_name` are audit-only.

In [10]:
weekly = builder.attach_metadata(weekly)
print("missing category:", weekly["category"].isna().mean())
weekly[["store_code", "product_code", "week_id", "price", "units", "category"]].head()

missing category: 0.0


,store_code,product_code,week_id,price,units,category
0,0,30,6,265.0000,15.0000,Кино - DVD
1,0,30,7,265.0000,13.0000,Кино - DVD
2,0,30,8,265.0000,3.0000,Кино - DVD
3,0,31,6,434.0000,7.0000,Кино - Blu-Ray
4,0,31,7,434.0000,2.0000,Кино - Blu-Ray


## 9. Markdown-price proxy

$$P^{ref}_{ist}=Q_{0.90}(P_{is,t-13},\ldots,P_{is,t-1})$$

$$\mathrm{on\_promo}=1[P_{ist}\le 0.95\,P^{ref}_{ist}]$$

Lookback is over **prior observed** store-product weeks (the panel is not dense).  
P90, not the max, so one spike does not label the next 13 weeks as promo.

Paper wording: *backward-looking markdown proxy*. Not *observed promotion*.

In [11]:
weekly = builder.construct_markdown_promo(weekly)
weekly[
    ["store_code", "product_code", "week_id", "price", "regular_price_ref", "promo_ref_available", "on_promo"]
].head(12)

promo_ref available: 0.5218689249334317 | on_promo rate among those: 0.260146653275514


,store_code,product_code,week_id,price,regular_price_ref,promo_ref_available,on_promo
0,0,1000,1,58.0000,NaN,False,NaN
1,0,1000,2,58.0000,NaN,False,NaN
2,0,1000,6,58.0000,NaN,False,NaN
3,0,1000,8,58.0000,NaN,False,NaN
4,0,10004,5,64.0000,NaN,False,NaN
5,0,1001,1,58.0000,NaN,False,NaN
6,0,10012,3,76.0000,NaN,False,NaN
7,0,10012,6,76.0000,NaN,False,NaN
8,0,10012,7,76.0000,NaN,False,NaN
9,0,1002,1,58.0000,NaN,False,NaN


## 10. Save the weekly master

All valid positive shop-item-weeks. **Do not delete this file.**

In [12]:
builder.weekly = builder.save_weekly_master(weekly)
builder.weekly.head()

Wrote /home/thebigmonster/Github/nn-elasticity-additional-work/data/predict-future-sales-1c/panel/1c_weekly_master.parquet


,store_code,product_code,week_id,week_start,units,revenue,n_sales_days,n_price_points,min_daily_price,max_daily_price,price,return_units,n_return_rows,item_name,item_category_id,category,shop_name,regular_price_ref,promo_ref_available,on_promo
0,0,1000,1,2013-01-07,3.0000,174.0000,3,1,58.0000,58.0000,58.0000,0.0000,0.0000,"3D Action Puzzle ""Зомби"" Уборщик",67,Подарки - Развитие,"!Якутск Орджоникидзе, 56 фран",NaN,False,NaN
1,0,1001,1,2013-01-07,1.0000,58.0000,1,1,58.0000,58.0000,58.0000,0.0000,0.0000,"3D Action Puzzle ""Зомби"" Шахтер",67,Подарки - Развитие,"!Якутск Орджоникидзе, 56 фран",NaN,False,NaN
2,0,1002,1,2013-01-07,1.0000,58.0000,1,1,58.0000,58.0000,58.0000,0.0000,0.0000,"3D Action Puzzle ""Техника"" Бомбардировщик",67,Подарки - Развитие,"!Якутск Орджоникидзе, 56 фран",NaN,False,NaN
3,0,1003,1,2013-01-07,1.0000,58.0000,1,1,58.0000,58.0000,58.0000,0.0000,0.0000,"3D Action Puzzle ""Техника"" Вертолет",67,Подарки - Развитие,"!Якутск Орджоникидзе, 56 фран",NaN,False,NaN
4,0,10063,1,2013-01-07,1.0000,69.0000,1,1,69.0000,69.0000,69.0000,0.0000,0.0000,ВРЕМЯ,40,Кино - DVD,"!Якутск Орджоникидзе, 56 фран",NaN,False,NaN


## 11. Selection window (no lookahead)

The second half of weeks must not influence stores, category or SKUs.

In [13]:
selection = builder.selection_sample()
print("rows:", len(selection), "weeks:", builder.n_selection_weeks)

Selection cutoff: 73 of 1 → 146
rows: 1318059 weeks: 73


## 12. Core stores

Keep shops with ≥85% week coverage, then the 20 most active. Starting universe, not necessarily final.

In [14]:
core_stores = builder.select_core_stores(selection)
selection_core = selection[selection["store_code"].isin(core_stores)].copy()
print("core rows:", len(selection_core))

core stores: ['31', '25', '54', '28', '57', '27', '42', '6', '50', '56', '58', '30', '46', '19', '15', '7', '16', '35', '26', '29']
core rows: 824881


## 13. Product diagnostics

`coverage_rate` = share of **core store-weeks** with an observed purchase.  
That is not shelf availability (unlike M5).

In [15]:
product_stats = builder.compute_product_stats(selection_core)
product_stats.head(20)

Wrote /home/thebigmonster/Github/nn-elasticity-additional-work/data/predict-future-sales-1c/panel/1c_product_diagnostics.csv


,product_code,item_category_id,category,n_obs,n_stores,n_weeks,total_units,total_revenue,unique_prices,mean_price,std_price,median_units,max_units,return_units,promo_ref_rows,promo_rate,price_cv,coverage_rate,promo_ref_coverage,return_rate
0,100,40,Кино - DVD,40,15,27,40.0000,"5,960.0000",1,149.0000,0.0000,1.0000,1.0000,0.0000,8,0.0000,0.0000,0.0274,0.2000,0.0000
1,1000,67,Подарки - Развитие,118,18,57,142.0000,"13,338.0000",2,93.6610,11.5235,1.0000,5.0000,0.0000,58,0.3448,0.1230,0.0808,0.4915,0.0000
2,10000,37,Кино - Blu-Ray,19,8,16,19.0000,"4,681.0000",2,246.3684,112.3903,1.0000,1.0000,0.0000,3,0.0000,0.4562,0.0130,0.1579,0.0000
3,10001,38,Кино - Blu-Ray 3D,31,15,19,33.0000,"11,516.7998",2,348.9935,0.0250,1.0000,2.0000,0.0000,1,0.0000,0.0001,0.0212,0.0323,0.0000
4,10002,40,Кино - DVD,51,17,16,72.0000,"28,727.1992",4,398.9905,0.0279,1.0000,6.0000,0.0000,6,0.0000,0.0001,0.0349,0.1176,0.0000
5,10004,40,Кино - DVD,62,16,38,63.0000,"9,386.7002",2,148.9952,0.0216,1.0000,2.0000,0.0000,20,0.0000,0.0001,0.0425,0.3226,0.0000
6,10005,37,Кино - Blu-Ray,1,1,1,1.0000,299.0000,1,299.0000,NaN,1.0000,1.0000,0.0000,0,NaN,NaN,0.0007,0.0000,0.0000
7,10006,41,Кино - Коллекционное,35,17,12,42.0000,"16,758.0000",1,399.0000,0.0000,1.0000,3.0000,0.0000,0,NaN,0.0000,0.0240,0.0000,0.0000
8,10007,41,Кино - Коллекционное,40,17,17,52.0000,"20,748.0000",1,399.0000,0.0000,1.0000,4.0000,0.0000,4,0.0000,0.0000,0.0274,0.1000,0.0000
9,10008,40,Кино - DVD,32,14,26,32.0000,"4,665.8999",3,145.8094,12.5419,1.0000,1.0000,0.0000,3,0.3333,0.0860,0.0219,0.0938,0.0000


## 14. Eligible screen

Density, price variation, promo-reference support, and a cap on return rate.  
Problematic UPCs are dropped as products, not winsorized row by row.

In [16]:
eligible = builder.screen_eligible(product_stats)

eligible SKUs: 135
      product_code                                category  coverage_rate  n_stores  n_weeks  unique_prices  price_cv  promo_rate  \
6313          1830           Игры PC - Стандартные издания         0.7479        20       73            111    0.1975      0.2480   
11751         6466           Игры PC - Стандартные издания         0.7384        20       73             83    0.2471      0.2956   
10933         5272           Игры PC - Стандартные издания         0.7068        20       73             76    0.3646      0.3676   
12634         7894                   Аксессуары - XBOX 360         0.6911        20       73             53    0.0449      0.1475   
9093          2308           Игры PC - Стандартные издания         0.6884        20       73             49    0.3684      0.5243   
11945         6740           Игры PC - Стандартные издания         0.6719        20       73             25    0.1349      0.1299   
2542         13071                        Аксессуа

## 15. Rank categories — STOP

Look at this table before freezing. Skip gift cards, services, face-value items, etc.

Do **not** choose a category from ICDN vs MLP results.  
Do **not** assume `item_category_id == 40`.

In [17]:
category_stats = builder.rank_categories(eligible)
category_stats

    item_category_id                          category  n_products  median_coverage  median_price_cv  median_n_stores  total_units
8                 30     Игры PC - Стандартные издания          38           0.4325           0.2755          20.0000  60,449.0000
7                 28  Игры PC - Дополнительные издания          11           0.3842           0.2602          20.0000  14,346.0000
3                 19                        Игры - PS3          21           0.3644           0.2994          20.0000  20,409.0000
12                40                        Кино - DVD          12           0.3271           0.2020          20.0000  12,579.0000
6                 23                   Игры - XBOX 360          20           0.3243           0.2645          20.0000  17,970.0000


,item_category_id,category,n_products,median_coverage,median_price_cv,median_n_stores,total_units
8,30,Игры PC - Стандартные издания,38,0.4325,0.2755,20.0000,"60,449.0000"
7,28,Игры PC - Дополнительные издания,11,0.3842,0.2602,20.0000,"14,346.0000"
3,19,Игры - PS3,21,0.3644,0.2994,20.0000,"20,409.0000"
12,40,Кино - DVD,12,0.3271,0.2020,20.0000,"12,579.0000"
6,23,Игры - XBOX 360,20,0.3243,0.2645,20.0000,"17,970.0000"


## 16. Freeze 20 SKUs by Jaccard co-occurrence

Greedy rule: start from the densest UPC, then add SKUs that mix own coverage with mean Jaccard against the set already chosen.

Cross-elasticities need \(P_i\) and \(P_j\) in the same store-week.

In [18]:
# Set this AFTER inspecting category_stats. Example 40 is not a default.
TARGET_CATEGORY_ID = int(category_stats.iloc[0]["item_category_id"])
print("Using item_category_id =", TARGET_CATEGORY_ID, "— change this if the top row is economically wrong.")

selection = builder.freeze_universe(TARGET_CATEGORY_ID)
selection.to_dict()

Using item_category_id = 30 — change this if the top row is economically wrong.
SELECTED_PRODUCTS: ['1830', '6466', '5272', '2308', '6740', '2445', '6457', '6738', '7070', '3325']
count   1,442.0000
mean        0.6469
std         0.2122
min         0.0000
25%         0.5000
50%         0.6000
75%         0.8000
max         1.0000
dtype: float64
store-weeks >= 8/10: 0.3418862690707351
store-weeks = 10/10: 0.07142857142857142
Wrote frozen store/SKU lists under /home/thebigmonster/Github/nn-elasticity-additional-work/data/predict-future-sales-1c/panel


{'cutoff_week_id': 73,
 'category_id': 30,
 'category': 'Игры PC - Стандартные издания',
 'core_stores': ['31',
  '25',
  '54',
  '28',
  '57',
  '27',
  '42',
  '6',
  '50',
  '56',
  '58',
  '30',
  '46',
  '19',
  '15',
  '7',
  '16',
  '35',
  '26',
  '29'],
 'candidate_product_codes': ['1830',
  '6466',
  '5272',
  '2308',
  '6740',
  '2445',
  '6457',
  '7070',
  '6738',
  '3325',
  '15044',
  '2252',
  '15063',
  '1470',
  '4779',
  '2753',
  '4809',
  '2929',
  '2833',
  '15016',
  '7912',
  '4790',
  '4370',
  '3186',
  '2854',
  '5260',
  '3432',
  '19415',
  '6121',
  '4723'],
 'product_codes': ['1830',
  '6466',
  '5272',
  '2308',
  '6740',
  '2445',
  '6457',
  '6738',
  '7070',
  '3325'],
 'n_eligible_in_category': 38,
 'joint_coverage_ge': 0.3418862690707351,
 'joint_coverage_all': 0.07142857142857142,
 'criteria': {'selection_frac': 0.5,
  'min_store_week_coverage': 0.85,
  'n_core_stores': 20,
  'min_stores': 10,
  'min_week_coverage': 0.7,
  'min_unique_prices': 8,
 

## 17. ICDN panel

Full horizon, frozen stores and SKUs, rows with `promo_ref_available` only.

Main spec does not fill missing promo. Robustness: `OneCWeeklyPanelBuilder.zero_promo_copy`.

In [19]:
icdn_panel = builder.build_icdn_panel()
print(icdn_panel.shape)
print(icdn_panel.dtypes)
icdn_panel.head(12)

rows dropped without promo reference: 800
rows dropped for non-positive price: 0
Wrote /home/thebigmonster/Github/nn-elasticity-additional-work/data/predict-future-sales-1c/panel/1c_icdn_panel.parquet
(12611, 7)
store_code      string[python]
product_code    string[python]
week_id                  int32
price                  float32
units                  float32
on_promo                  int8
category        string[python]
dtype: object


,store_code,product_code,week_id,price,units,on_promo,category
0,15,1830,5,599.0000,4.0000,0,Игры PC - Стандартные издания
1,15,2308,5,799.0000,2.0000,0,Игры PC - Стандартные издания
2,15,3325,5,799.0000,7.0000,1,Игры PC - Стандартные издания
3,15,5272,5,299.5000,8.0000,1,Игры PC - Стандартные издания
4,15,6740,5,499.0000,2.0000,0,Игры PC - Стандартные издания
5,16,1830,5,599.0000,8.0000,0,Игры PC - Стандартные издания
6,16,3325,5,799.0000,3.0000,1,Игры PC - Стандартные издания
7,16,5272,5,299.5000,1.0000,1,Игры PC - Стандартные издания
8,19,1830,5,599.0000,9.0000,0,Игры PC - Стандартные издания
9,19,2308,5,799.0000,4.0000,0,Игры PC - Стандартные издания


## 18. Final checks

In [20]:
print("price min / units min :", icdn_panel["price"].min(), icdn_panel["units"].min())
print("on_promo values       :", sorted(icdn_panel["on_promo"].unique().tolist()))
print("n products            :", icdn_panel["product_code"].nunique())
print("n stores              :", icdn_panel["store_code"].nunique())
print("week_id range         :", icdn_panel["week_id"].min(), "→", icdn_panel["week_id"].max())
print("outputs:", sorted(p.name for p in OUT_DIR.iterdir()))
icdn_panel.groupby("product_code", observed=True).agg(
    n_obs=("units", "size"),
    n_weeks=("week_id", "nunique"),
    n_stores=("store_code", "nunique"),
    mean_units=("units", "mean"),
    promo_rate=("on_promo", "mean"),
)

price min / units min : 0.5 1.0
on_promo values       : [0, 1]
n products            : 10
n stores              : 20
week_id range         : 5 → 146
outputs: ['1c_category_diagnostics.csv', '1c_icdn_panel.parquet', '1c_product_diagnostics.csv', '1c_returns_audit.parquet', '1c_sales_audit.json', '1c_selected_products.csv', '1c_selected_skus.json', '1c_selected_stores.csv', '1c_store_diagnostics.csv', '1c_weekly_master.parquet', 'icdn']


,n_obs,n_weeks,n_stores,mean_units,promo_rate
product_code,,,,,
1830,1291,130,20,2.7382,0.1960
2308,1796,142,20,2.0161,0.4053
2445,1404,135,20,2.3675,0.4181
3325,775,97,20,1.9819,0.5884
5272,1454,138,20,2.1657,0.2854
6457,1320,133,20,2.0348,0.4326
6466,1085,87,20,2.9539,0.2774
6738,1077,129,20,1.9889,0.3203
6740,1379,137,20,2.5011,0.1139
